<a href="https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/MoharanaSudhanshu/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 168 (delta 82), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.90 MiB | 11.55 MiB/s, done.
Resolving deltas: 100% (82/82), done.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import glob
import pandas as pd
import numpy as np

# ============================================================
# 1. FIND AND LOAD A DATASET
# ============================================================

# Search for common dataset files
csv_files = glob.glob("**/*.csv", recursive=True)
parquet_files = glob.glob("**/*.parquet", recursive=True)

print("CSV files found:", csv_files[:10])
print("Parquet files found:", parquet_files[:10])

if csv_files:
    DATA_PATH = csv_files[0]
    df = pd.read_csv(DATA_PATH)
    print(f"\nLoaded CSV: {DATA_PATH}")

elif parquet_files:
    DATA_PATH = parquet_files[0]
    df = pd.read_parquet(DATA_PATH)
    print(f"\nLoaded Parquet: {DATA_PATH}")

else:
    raise FileNotFoundError(
        "No CSV or Parquet dataset was found. "
        "Update DATA_PATH with the correct dataset location."
    )

# ============================================================
# 2. BASIC DATA INSPECTION
# ============================================================

print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

# ============================================================
# 3. BUILD FEATURE VECTOR
# ============================================================

# Remove completely empty columns
df = df.dropna(axis=1, how="all")

# Separate numerical and categorical features
numeric_features = df.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

# Fill missing numeric values with median
for col in numeric_features:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with "missing"
for col in categorical_features:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna("missing")

# One-hot encode categorical features
feature_vector = pd.get_dummies(
    df,
    columns=categorical_features,
    drop_first=False
)

# Keep numeric values
feature_vector = feature_vector.replace(
    [np.inf, -np.inf],
    np.nan
)

feature_vector = feature_vector.fillna(0)

print("\nFinal feature vector shape:")
print(feature_vector.shape)

print("\nFeature vector preview:")
display(feature_vector.head())

CSV files found: ['flyrank-ml-internship/data/raw/content_refresh_anonymized.csv', 'flyrank-ml-internship/outputs/refresh_queue_sample.csv', 'sample_data/california_housing_test.csv', 'sample_data/mnist_train_small.csv', 'sample_data/california_housing_train.csv', 'sample_data/mnist_test.csv']
Parquet files found: []

Loaded CSV: flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_updat

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Data types:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
a

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,trend_direction_down,trend_direction_flat,trend_direction_new,trend_direction_stable,trend_direction_up
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,16,...,False,False,False,True,False,True,False,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,False,False,True,False,False,True,False,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,11,...,False,False,True,False,False,True,False,False,False,False
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,87,78,75,...,False,True,False,False,False,False,False,False,True,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,144,...,False,False,True,False,False,True,False,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

The feature vector contains numerical and categorical variables available in the dataset.

**Numerical features:** Missing numerical values are replaced with the median value of the corresponding feature. Median imputation is used because it is less sensitive to extreme values than mean imputation.

**Categorical features:** Missing categorical values are replaced with the category `missing`. Categorical variables are then converted into numerical representations using one-hot encoding.

**Availability check:** Only features that are available before the prediction or decision point should be included in the final model. Features generated after the outcome occurs, future performance values, or direct representations of the target should be excluded.

The final feature vector is checked for missing and infinite values before it is used for further analysis.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# FEATURE NOTES AND DATA QUALITY CHECK
# ============================================================

feature_summary = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean() * 100
    ).round(2).values,
    "unique_values": df.nunique().values
})

def classify_feature(column):
    dtype = df[column].dtype

    if pd.api.types.is_numeric_dtype(dtype):
        return "numerical"

    elif (
        pd.api.types.is_object_dtype(dtype)
        or pd.api.types.is_categorical_dtype(dtype)
    ):
        return "categorical"

    else:
        return "other"


feature_summary["feature_type"] = [
    classify_feature(col)
    for col in df.columns
]

print("Feature summary:")
display(feature_summary)

print("\nMissing values after preprocessing:")
print(feature_vector.isna().sum().sum())

print("\nFinal number of features:")
print(feature_vector.shape[1])

print("\nFinal number of observations:")
print(feature_vector.shape[0])

Feature summary:


,feature,dtype,missing_values,missing_percentage,unique_values,feature_type
content_id,content_id,object,0,0.0,30000,categorical
client_id,client_id,object,0,0.0,32,categorical
search_volume,search_volume,float64,0,0.0,41,numerical
competition,competition,float64,0,0.0,101,numerical
competition_level,competition_level,object,0,0.0,4,categorical
cpc,cpc,float64,0,0.0,915,numerical
content_type,content_type,object,0,0.0,3,categorical
main_intent,main_intent,object,0,0.0,5,categorical
word_count,word_count,float64,0,0.0,5476,numerical
char_count,char_count,float64,0,0.0,14839,numerical



Missing values after preprocessing:
0

Final number of features:
30115

Final number of observations:
30000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt

Potential data leakage was investigated by checking column names and feature values for signals that may contain information unavailable at prediction time.

Particular attention was given to:

- target or label-derived columns,
- future performance variables,
- outcome-related fields,
- status fields created after the prediction event,
- identifiers that may indirectly encode the target.

Potentially suspicious features are reported for manual review. A feature should only remain in the model if it is genuinely available before the prediction is made.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# LEAKAGE HUNT
# ============================================================

# Common words associated with possible leakage
leakage_keywords = [
    "target",
    "label",
    "outcome",
    "result",
    "future",
    "conversion",
    "converted",
    "revenue",
    "after",
    "post",
    "final",
    "success",
    "winner"
]

# Search column names
suspicious_columns = []

for column in df.columns:

    column_lower = column.lower()

    for keyword in leakage_keywords:

        if keyword in column_lower:
            suspicious_columns.append(column)
            break


print("Possible leakage-related columns:")
print(suspicious_columns)

# ============================================================
# HIGH-CARDINALITY / IDENTIFIER CHECK
# ============================================================

high_cardinality_columns = []

for column in df.columns:

    unique_ratio = (
        df[column].nunique() / len(df)
    )

    if unique_ratio > 0.90:
        high_cardinality_columns.append(column)


print("\nPossible identifier/high-cardinality columns:")
print(high_cardinality_columns)

# ============================================================
# CHECK FOR DUPLICATE ROWS
# ============================================================

duplicate_rows = df.duplicated().sum()

print("\nDuplicate rows:")
print(duplicate_rows)

# ============================================================
# CREATE REVIEW TABLE
# ============================================================

review_columns = sorted(
    set(suspicious_columns + high_cardinality_columns)
)

leakage_review = pd.DataFrame({
    "column": review_columns
})

if len(review_columns) > 0:

    leakage_review["unique_values"] = [
        df[col].nunique()
        for col in review_columns
    ]

    leakage_review["missing_values"] = [
        df[col].isna().sum()
        for col in review_columns
    ]

    print("\nColumns requiring manual leakage review:")
    display(leakage_review)

else:
    print(
        "\nNo columns were automatically flagged "
        "by the keyword-based leakage check."
    )

print("\nLeakage check completed successfully.")

Possible leakage-related columns:
[]

Possible identifier/high-cardinality columns:
['content_id']

Duplicate rows:
0

Columns requiring manual leakage review:


,column,unique_values,missing_values
0,content_id,30000,0



Leakage check completed successfully.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### What I Excluded and Why

Potential leakage features were excluded when they could reveal information about the outcome after the prediction point.

The following types of features should not be used:

1. **Target-derived columns** — These directly contain or strongly encode the value being predicted.
2. **Future information** — These are not available when the prediction is made.
3. **Post-outcome fields** — These are created after the result has already occurred.
4. **Identifiers** — Unique IDs may not generalize and can sometimes indirectly encode information.
5. **Private or sensitive information** — These should be excluded unless explicitly required and properly justified.

The final feature set should contain only variables that are available before the prediction point and are appropriate for the intended decision-support task.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# EXCLUSION CHECK
# ============================================================

excluded_columns = sorted(
    set(suspicious_columns + high_cardinality_columns)
)

print("Columns proposed for exclusion:")
print(excluded_columns)

# Create a safer feature set
safe_features = df.drop(
    columns=excluded_columns,
    errors="ignore"
)

print("\nOriginal dataset shape:")
print(df.shape)

print("\nDataset shape after exclusions:")
print(safe_features.shape)

print("\nRemaining columns:")
print(safe_features.columns.tolist())

# ============================================================
# FINAL CHECK
# ============================================================

print("\nFinal checks:")

print(
    "Remaining missing values:",
    safe_features.isna().sum().sum()
)

print(
    "Duplicate rows:",
    safe_features.duplicated().sum()
)

print(
    "Number of remaining features:",
    safe_features.shape[1]
)

print("\nFeature leakage/privacy check completed.")

Columns proposed for exclusion:
['content_id']

Original dataset shape:
(30000, 44)

Dataset shape after exclusions:
(30000, 43)

Remaining columns:
['client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Final checks:
Remaining missing values: 0
Duplicate rows: 0
Number of remaining features: 43


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.